# Rec-System Playground

Этот ноутбук один в один повторяет пайплайн рекомендательной системы из `rec-system/`:

1. **Content Pipeline** — загрузка сырого контента из CSV → полный NLP pipeline (topics, sentiment, NER, embeddings, text metrics) → `ContentFeatures`.
2. **Onboarding** — создание N пользователей с разными темами-предпочтениями, инициализация `UserProfile`.
3. **Feed Generation** — двухэтапный retrieval (freshness + embedding) → dedup clustering (stub) → 6-компонентный скоринг → diversity filter → финальный порядок UUID.
4. **Interactions** — симуляция поведения фронта: IMPRESSION / OPEN / CLOSE / LIKE / DISLIKE / BOOKMARK → `SignalClassifier` → `ProfileUpdater` (EMA).
5. **Journey Report** — что увидел каждый пользователь, в каком порядке, как менялся его профиль.
6. **Experiments** — меняем `max_topic_ratio`, `max_topic_streak`, `scoring_weights` и видим разницу.

Все формулы, веса и пороги **ТОЧЬ-В-ТОЧЬ** скопированы из `rec-system/src/` (см. сноски в каждой секции).

---

**Как запускать:**

```bash
cd /home/mattew/SKD/rec-system
uv run jupyter notebook /home/mattew/SKD/rec_playground.ipynb
# или для headless прогона:
uv run jupyter nbconvert --to notebook --execute /home/mattew/SKD/rec_playground.ipynb
```

Модели (`rubert-tiny2`, `rubert-nli-threeway`, `rubert-sentiment`, `ru_core_news_lg`) уже закешированы на диске — первая ячейка с загрузкой займёт 30-60 секунд.


## 1. Environment setup

Добавляем `rec-system/` в `sys.path` только для удобства (классы ниже — автономные копии, не импортируют `src.*`).
Этот подход позволяет запускать ноутбук из `rec-system/.venv` и переиспользовать закешированные HuggingFace модели.


In [1]:
import json
import math
import pickle
import random
import re
import sys
import time
import uuid
from collections import defaultdict
from dataclasses import dataclass, field, replace
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Optional, Set

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/home/mattew/SKD")
REC_SYSTEM_ROOT = PROJECT_ROOT / "rec-system"
CACHE_DIR = PROJECT_ROOT / ".playground_cache"
CACHE_DIR.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"PROJECT_ROOT   = {PROJECT_ROOT}")
print(f"REC_SYSTEM     = {REC_SYSTEM_ROOT}")
print(f"CACHE_DIR      = {CACHE_DIR}")


PROJECT_ROOT   = /home/mattew/SKD
REC_SYSTEM     = /home/mattew/SKD/rec-system
CACHE_DIR      = /home/mattew/SKD/.playground_cache


## 2. Configuration (mirror of `rec_config` table defaults)

Все значения взяты из `rec-system/src/infrastructure/config/rec_config_loader.py`.
Меняйте эти константы, чтобы проверять влияние настроек.


In [2]:
# === signal_weights (rec_config_loader.py:7-18) ===
SIGNAL_WEIGHTS: dict = {
    "impression_read": {"threshold_duration_ms": 2000, "weight": 0.15},
    "impression_skip": {"threshold_duration_ms": 2000, "weight": -0.05},
    "open": {"weight": 0.1},
    "close_fast": {"threshold_duration_ms": 3000, "weight": -0.2},
    "close_full": {"threshold_scroll_pct": 0.85, "threshold_duration_ms": 15000, "weight": 0.5},
    "close_half": {"threshold_scroll_pct": 0.5, "threshold_duration_ms": 10000, "weight": 0.4},
    "close_other": {"weight": 0.1},
    "like": {"weight": 0.6},
    "dislike": {"weight": -0.7},
    "bookmark": {"weight": 0.8},
    "orphan_timeout_minutes": 5,
}

# === scoring_weights (rec_config_loader.py:20-27) ===
SCORING_WEIGHTS: dict = {
    "topic_match": 0.30,
    "embedding_sim": 0.25,
    "entity_match": 0.15,
    "sentiment_match": 0.05,
    "freshness": 0.15,
    "format_match": 0.10,
}

# === profile_params (rec_config_loader.py:29-38) ===
PROFILE_PARAMS: dict = {
    "learning_rate": 0.08,
    "entity_min_weight": 0.4,
    "entity_max_per_post": 5,
    "entity_cleanup_days": 30,
    "entity_decay_factor": 0.95,
    "job_interval_minutes": 5,
    "job_batch_threshold": 20,
    "open_orphan_timeout_minutes": 5,
}

# === ranking_params (rec_config_loader.py:40-49) ===
RANKING_PARAMS: dict = {
    "freshness_halflife_hours": 48,
    "candidate_freshness_limit": 300,
    "candidate_embedding_limit": 200,
    "feed_size": 200,
    "page_size": 30,
    "max_topic_streak": 3,
    "max_topic_ratio": 0.40,
    "candidate_max_age_days": 7,
}

# === onboarding_params (rec_config_loader.py:51-55) ===
ONBOARDING_PARAMS: dict = {
    "baseline_weight": 0.01,
    "min_topics": 3,
    "max_topics": 5,
}

# === dedup_params (generate_feed.py + design/09-configuration.md) ===
DEDUP_PARAMS: dict = {
    "enable_dedup_clustering": True,
    "enable_related_spacing": True,
    "related_min_gap": 3,
}

# === Aggregate config dict (как передаётся во все сервисы в production) ===
# Используется внутри сервисов через `config.get(...)`. Обратите внимание: в prod эти ключи
# лежат в разных группах, но в use-case'ах (см. generate_feed.py) мержатся в один dict.
FEED_CONFIG: dict = {
    **SCORING_WEIGHTS,
    "scoring_weights": SCORING_WEIGHTS,
    **RANKING_PARAMS,
    "max_age_hours": RANKING_PARAMS["candidate_max_age_days"] * 24,
    "freshness_halflife_hours": RANKING_PARAMS["freshness_halflife_hours"],
    "freshness_candidate_limit": RANKING_PARAMS["candidate_freshness_limit"],
    "embedding_candidate_limit": RANKING_PARAMS["candidate_embedding_limit"],
    **PROFILE_PARAMS,
}

print("Config loaded. Tunable parameters ready.")


Config loaded. Tunable parameters ready.


## 3. Value Objects (verbatim copies from `rec-system/src/domain/value_objects/`)

- `TopicVector` — 18 тем, L1-нормализация, формула `new[t] += lr * weight * topic_score`.
- `SentimentPrefs` — 3 класса, формула `new[s] += lr * weight * 0.5`.
- `FormatPrefs` — EMA по `word_count` и `complexity`.


In [3]:
# === TopicVector (domain/value_objects/topic_vector.py) ===
TOPICS: list[str] = [
    "политика", "экономика", "технологии", "наука", "спорт", "культура",
    "общество", "происшествия", "международные новости", "бизнес",
    "финансы", "образование", "здоровье", "развлечения", "криминал",
    "армия", "природа", "транспорт",
]


class TopicVector:
    """L1-normalized distribution over 18 topics. Immutable."""

    def __init__(self, weights: dict[str, float], topics: list[str] | None = None) -> None:
        active = topics if topics is not None else TOPICS
        clamped = {t: max(0.0, weights.get(t, 0.0)) for t in active}
        total = sum(clamped.values())
        if total == 0.0:
            self._w = {t: 1.0 / len(active) for t in active}
        else:
            self._w = {t: v / total for t, v in clamped.items()}

    @property
    def weights(self) -> dict[str, float]:
        return dict(self._w)

    def get(self, topic: str) -> float:
        return self._w.get(topic, 0.0)

    @classmethod
    def create_from_selected(
        cls,
        selected: list[str],
        baseline_weight: float = 0.01,
        all_topics: list[str] | None = None,
    ) -> "TopicVector":
        active = all_topics if all_topics is not None else TOPICS
        weights = {t: baseline_weight for t in active}
        for t in selected:
            if t in weights:
                weights[t] = 1.0  # равная доля, normalization приведёт к 1/len(selected)
        return cls(weights, topics=active)

    def apply_update(self, lr: float, weight: float, topic_scores: dict[str, float]) -> "TopicVector":
        new = dict(self._w)
        for t, s in topic_scores.items():
            if t in new:
                new[t] += lr * weight * s
        return TopicVector(new, topics=list(self._w.keys()))

    def to_dict(self) -> dict[str, float]:
        return dict(self._w)


# === SentimentPrefs (domain/value_objects/sentiment_prefs.py) ===
SENTIMENTS = ["POSITIVE", "NEGATIVE", "NEUTRAL"]


class SentimentPrefs:
    """L1-normalized distribution over POSITIVE/NEGATIVE/NEUTRAL."""

    def __init__(self, weights: dict[str, float]) -> None:
        clamped = {s: max(0.0, weights.get(s, 0.0)) for s in SENTIMENTS}
        total = sum(clamped.values())
        if total == 0.0:
            self._w = {s: 1.0 / 3 for s in SENTIMENTS}
        else:
            self._w = {s: v / total for s, v in clamped.items()}

    @property
    def weights(self) -> dict[str, float]:
        return dict(self._w)

    def get(self, s: str) -> float:
        return self._w.get(s, 0.0)

    @classmethod
    def create_uniform(cls) -> "SentimentPrefs":
        return cls({s: 1.0 for s in SENTIMENTS})

    def apply_update(self, sentiment: str, lr: float, weight: float) -> "SentimentPrefs":
        # Формула из prod: new[s] += lr * weight * 0.5
        new = dict(self._w)
        if sentiment in new:
            new[sentiment] += lr * weight * 0.5
        return SentimentPrefs(new)

    def to_dict(self) -> dict[str, float]:
        return dict(self._w)


# === FormatPrefs (domain/value_objects/format_prefs.py) ===
class FormatPrefs:
    """EMA-updated format preferences: avg_length (words) and avg_complexity."""

    def __init__(self, avg_length: float, avg_complexity: float) -> None:
        self.avg_length = avg_length
        self.avg_complexity = avg_complexity

    @classmethod
    def create_default(cls) -> "FormatPrefs":
        return cls(avg_length=200.0, avg_complexity=0.5)

    def apply_ema_update(self, lr: float, post_length: float, post_complexity: float) -> "FormatPrefs":
        new_length = (1 - lr) * self.avg_length + lr * post_length
        new_complexity = (1 - lr) * self.avg_complexity + lr * post_complexity
        return FormatPrefs(avg_length=new_length, avg_complexity=new_complexity)

    def to_dict(self) -> dict[str, float]:
        return {"avg_length": self.avg_length, "avg_complexity": self.avg_complexity}


# Быстрый sanity-check
tv = TopicVector.create_from_selected(["технологии", "наука", "здоровье"])
print("Sample TopicVector (3 selected of 18):")
for t, w in sorted(tv.weights.items(), key=lambda kv: -kv[1])[:5]:
    print(f"  {t:30s} {w:.4f}")


Sample TopicVector (3 selected of 18):
  технологии                     0.3175
  наука                          0.3175
  здоровье                       0.3175
  политика                       0.0032
  экономика                      0.0032


## 4. Entities: `ContentFeatures`, `UserProfile`, `UserInteraction`, `Signal`

Полный маппинг схемы таблиц `posts_features`, `rec_profiles`, `rec_entity_interests`, `user_interactions`.


In [4]:
# === ContentFeatures (domain/entities/content_features.py) ===
# dataclass для краткости — поведение идентично prod-версии.
@dataclass
class ContentFeatures:
    post_id: uuid.UUID
    content: str
    post_date: datetime
    title: Optional[str] = None
    source_id: Optional[uuid.UUID] = None
    source_type: Optional[str] = None
    text_length: Optional[int] = None
    word_count: Optional[int] = None
    reading_time: Optional[float] = None
    complexity: Optional[float] = None
    is_short_form: Optional[bool] = None
    is_long_form: Optional[bool] = None
    topic_1: Optional[str] = None
    topic_1_score: Optional[float] = None
    topic_2: Optional[str] = None
    topic_2_score: Optional[float] = None
    topic_3: Optional[str] = None
    topic_3_score: Optional[float] = None
    sentiment: Optional[str] = None
    sentiment_score: Optional[float] = None
    entities_persons: list[str] = field(default_factory=list)
    entities_organizations: list[str] = field(default_factory=list)
    entities_locations: list[str] = field(default_factory=list)
    embedding: Optional[list[float]] = None
    processed_at: Optional[datetime] = None

    def topic_dict(self) -> dict[str, float]:
        r: dict[str, float] = {}
        if self.topic_1 and self.topic_1_score is not None:
            r[self.topic_1] = self.topic_1_score
        if self.topic_2 and self.topic_2_score is not None:
            r[self.topic_2] = self.topic_2_score
        if self.topic_3 and self.topic_3_score is not None:
            r[self.topic_3] = self.topic_3_score
        return r

    def all_entities(self) -> list[str]:
        return list(self.entities_persons) + list(self.entities_organizations) + list(self.entities_locations)


# === UserProfile (domain/entities/user_profile.py) ===
@dataclass
class UserProfile:
    user_id: uuid.UUID
    topic_vector: TopicVector
    sentiment_prefs: SentimentPrefs
    format_prefs: FormatPrefs
    interaction_count: int = 0
    created_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    last_updated: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    embedding: Optional[list[float]] = None
    cold_start: bool = True

    def copy_with(self, **kw) -> "UserProfile":
        return replace(self, **kw)


# === EventType / Signal / UserInteraction ===
class EventType:
    IMPRESSION = "IMPRESSION"
    OPEN = "OPEN"
    CLOSE = "CLOSE"
    LIKE = "LIKE"
    DISLIKE = "DISLIKE"
    BOOKMARK = "BOOKMARK"


@dataclass
class Signal:
    post_id: uuid.UUID
    user_id: uuid.UUID
    weight: float
    event_type: str

    @property
    def is_positive(self) -> bool:
        return self.weight > 0.0


@dataclass
class UserInteraction:
    id: int
    event_id: uuid.UUID
    user_id: uuid.UUID
    post_id: uuid.UUID
    event_type: str  # IMPRESSION | OPEN | CLOSE | LIKE | DISLIKE | BOOKMARK
    duration_ms: Optional[int]
    scroll_pct: Optional[float]
    max_scroll_pct: Optional[float]
    created_at: datetime
    processed: bool = False


print("Entities defined:", ContentFeatures.__name__, UserProfile.__name__, Signal.__name__)


Entities defined: ContentFeatures UserProfile Signal


## 5. Text window — Head + Middle + Tail truncation

Копия `rec-system/src/infrastructure/nlp/text_window.py`. Используется перед передачей текста
в BERT-модели (512 токенов). Делит бюджет: 50% head / 25% middle / 25% tail.


In [5]:
def prepend_title(title: str | None, body: str) -> str:
    """Префиксим заголовок к телу поста (text_window.py:70-86)."""
    if not title:
        return body
    title = title.strip()
    if not title:
        return body
    if not body:
        return title
    sep = ". " if not title.endswith((".", "!", "?", ":")) else " "
    return f"{title}{sep}{body}"


def extract_hmt(
    text: str,
    tokenizer,
    max_tokens: int,
    separator: str = " [...] ",
) -> tuple[str, bool]:
    """Token-aware Head+Middle+Tail (text_window.py:16-67)."""
    if not text:
        return "", False

    tokens = tokenizer.encode(text, add_special_tokens=False)
    n = len(tokens)
    if n <= max_tokens - 8:
        return text, False

    sep_tokens = tokenizer.encode(separator, add_special_tokens=False)
    sep_len = len(sep_tokens)
    budget = max_tokens - 8 - 2 * sep_len
    if budget <= 0:
        return tokenizer.decode(tokens[: max(1, max_tokens - 8)], skip_special_tokens=True), True

    head_budget = budget // 2
    tail_budget = budget // 4
    mid_budget = budget - head_budget - tail_budget

    head = tokens[:head_budget]
    mid_start = (n - mid_budget) // 2
    mid = tokens[mid_start : mid_start + mid_budget]
    tail = tokens[n - tail_budget :]

    return (
        tokenizer.decode(head, skip_special_tokens=True)
        + separator
        + tokenizer.decode(mid, skip_special_tokens=True)
        + separator
        + tokenizer.decode(tail, skip_special_tokens=True),
        True,
    )


print("H+M+T window function defined")


H+M+T window function defined


## 6. NLP Model Loading (singleton)

Одинаково с prod (`infrastructure/nlp/torch_*.py`):
| Задача | Модель | Размер | Batch | max_length |
|---|---|---|---|---|
| Topics (zero-shot NLI) | `cointegrated/rubert-base-cased-nli-threeway` | base | 32 | 512 |
| Sentiment | `blanchefort/rubert-base-cased-sentiment` | base | 128 | 512 |
| NER | `ru_core_news_lg` (spaCy) | lg | — | 4096 chars |
| Embeddings | `cointegrated/rubert-tiny2` (SentenceTransformer) | tiny | 32 | token-aware |

Первая загрузка занимает 30-60 сек. Все модели уже должны быть в `~/.cache/huggingface/hub/`.


In [6]:
import torch
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import spacy
import textstat

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch device: {DEVICE}")

# Singleton holders — модели грузятся один раз.
_MODELS: dict[str, Any] = {}


def load_topic_classifier():
    if "topic" not in _MODELS:
        print("Loading topic classifier (rubert-nli-threeway)...")
        _MODELS["topic"] = pipeline(
            "zero-shot-classification",
            model="cointegrated/rubert-base-cased-nli-threeway",
            device=DEVICE,
            torch_dtype=torch.bfloat16,
        )
    return _MODELS["topic"]


def load_sentiment_analyzer():
    if "sentiment" not in _MODELS:
        print("Loading sentiment analyzer (rubert-sentiment)...")
        _MODELS["sentiment"] = pipeline(
            "text-classification",
            model="blanchefort/rubert-base-cased-sentiment",
            device=DEVICE,
            torch_dtype=torch.bfloat16,
        )
    return _MODELS["sentiment"]


def load_ner():
    if "ner" not in _MODELS:
        print("Loading spaCy ru_core_news_lg...")
        nlp = spacy.load("ru_core_news_lg", disable=["lemmatizer", "attribute_ruler"])
        nlp.max_length = 5_000_000
        _MODELS["ner"] = nlp
    return _MODELS["ner"]


def load_encoder():
    if "encoder" not in _MODELS:
        print("Loading content encoder (rubert-tiny2)...")
        _MODELS["encoder"] = SentenceTransformer("cointegrated/rubert-tiny2", device=DEVICE)
    return _MODELS["encoder"]


# Прогреваем сразу — легче ловить ошибки тут, чем в pipeline ниже.
load_topic_classifier()
load_sentiment_analyzer()
load_ner()
load_encoder()
print("All 4 NLP models ready.")


Torch device: cuda
Loading topic classifier (rubert-nli-threeway)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-base-cased-nli-threeway
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading sentiment analyzer (rubert-sentiment)...


...

Loading spaCy ru_core_news_lg...


Loading content encoder (rubert-tiny2)...


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All 4 NLP models ready.


## 7. Text Analyzer (`infrastructure/nlp/text_analyzer_service.py`)

13 фичей: длина, число слов/предложений, unique_word_ratio, caps_ratio,
digits, ссылки, время чтения, is_long_form / is_short_form, complexity (Flesch-Oborneva).


In [7]:
textstat.set_lang("ru")

_READING_SPEED_WPM = 200
_LONG_FORM_THRESHOLD = 300
_SHORT_FORM_THRESHOLD = 50
_MIN_CHARS_FOR_COMPLEXITY = 10
_URL_RE = re.compile(r"https?://\S+")


def analyze_text(text: str) -> dict:
    """Копия TextAnalyzerService.analyze — строка в строку."""
    if not text:
        return {
            "text_length": 0, "word_count": 0, "sentence_count": 0,
            "avg_word_length": 0.0, "unique_word_ratio": 0.0, "caps_ratio": 0.0,
            "digit_count": 0, "has_links": False, "link_count": 0,
            "reading_time_min": 0.0, "is_long_form": False, "is_short_form": True,
            "complexity": 0.5,
        }
    words = text.split()
    wc = len(words)
    sentences = [s for s in text.split(".") if s.strip()]
    letters = [c for c in text if c.isalpha()]
    links = _URL_RE.findall(text)
    caps = sum(1 for c in letters if c.isupper())

    # complexity via textstat Flesch-Oborneva, нормализуем в [0,1]
    complexity = 0.5
    stripped = text.strip()
    if len(stripped) >= _MIN_CHARS_FOR_COMPLEXITY:
        try:
            fre = textstat.flesch_reading_ease(stripped)
            fre = max(0.0, min(100.0, fre))
            complexity = round((100.0 - fre) / 100.0, 4)
        except Exception:
            complexity = 0.5

    return {
        "text_length": len(text),
        "word_count": wc,
        "sentence_count": len(sentences),
        "avg_word_length": round(sum(len(w) for w in words) / wc, 2) if wc else 0.0,
        "unique_word_ratio": round(len(set(words)) / wc, 3) if wc else 0.0,
        "caps_ratio": round(caps / len(letters), 3) if letters else 0.0,
        "digit_count": sum(c.isdigit() for c in text),
        "has_links": len(links) > 0,
        "link_count": len(links),
        "reading_time_min": round(wc / _READING_SPEED_WPM, 2),
        "is_long_form": wc >= _LONG_FORM_THRESHOLD,
        "is_short_form": wc < _SHORT_FORM_THRESHOLD,
        "complexity": complexity,
    }


# Sanity
_demo = analyze_text("Это тестовый текст. Он содержит два предложения.")
print("Text analyzer sample:", {k: _demo[k] for k in ("word_count", "sentence_count", "complexity")})


Text analyzer sample: {'word_count': 7, 'sentence_count': 2, 'complexity': 0.1792}


## 8. NER extraction helper

Копия `SpacyEntityExtractor.extract` — фильтрация «мусорных» сущностей (< 2 символов, без букв).


In [8]:
_NER_LABEL_MAP = {"PER": "persons", "ORG": "organizations", "LOC": "locations"}
_NER_MAX_TEXT = 4096
_NER_MIN_LEN = 2
_HAS_LETTER = re.compile(r"[a-zA-Zа-яА-ЯёЁ]")


def _valid_entity(t: str) -> bool:
    s = t.strip()
    if len(s) < _NER_MIN_LEN:
        return False
    if not _HAS_LETTER.search(s):
        return False
    return True


def extract_entities(text: str) -> dict[str, list[str]]:
    out = {"persons": [], "organizations": [], "locations": []}
    if not text:
        return out
    nlp = load_ner()
    doc = nlp(text[:_NER_MAX_TEXT])
    seen = {k: set() for k in out}
    for ent in doc.ents:
        cat = _NER_LABEL_MAP.get(ent.label_)
        if not cat:
            continue
        etext = ent.text.strip()
        if not _valid_entity(etext):
            continue
        if etext not in seen[cat]:
            seen[cat].add(etext)
            out[cat].append(etext)
    return out


print("NER entity extractor ready")


NER entity extractor ready


## 9. Content Processing Pipeline (`ProcessContentUseCase._process_single`)

Для каждого поста:
1. `prepend_title(title, body)` — добавляем заголовок.
2. Byte guard: если body > 1_000_000 байт → truncate до 500_000 байт.
3. `extract_hmt(full_text, tokenizer=encoder.tokenizer, max_tokens=512)` — H+M+T усечение.
4. 4 NLP модели + text analyzer.
5. Собираем `ContentFeatures`.

Результат кешируется на диск (`.playground_cache/features_<csv_hash>.pkl`), чтобы не прогонять заново.


In [9]:
MAX_NLP_TOKENS = 512
MAX_TEXT_BYTES = 1_000_000
TRUNCATE_TEXT_BYTES = 500_000
HYPOTHESIS_TEMPLATE = "Этот текст про {}."
TOPIC_CLASSIFIER_TOP_K = 3


def process_single_post(post: dict) -> ContentFeatures:
    """Прогоняем один пост через весь NLP pipeline.

    post: dict c полями id, title, body, source_id, source_type, post_date.
    """
    post_id = uuid.UUID(post["id"]) if isinstance(post["id"], str) else post["id"]
    title = post.get("title") or ""
    body = post.get("body") or ""
    source_id = (
        uuid.UUID(post["source_id"]) if isinstance(post.get("source_id"), str) else post.get("source_id")
    )
    source_type = post.get("source_type") or "TELEGRAM"
    post_date = post.get("post_date")
    if isinstance(post_date, str):
        post_date = datetime.fromisoformat(post_date.replace("Z", "+00:00"))
    if post_date is None:
        post_date = datetime.now(timezone.utc)
    if post_date.tzinfo is None:
        post_date = post_date.replace(tzinfo=timezone.utc)

    # Byte guard (process_content.py:113-120)
    if len(body.encode("utf-8")) > MAX_TEXT_BYTES:
        body = body[:TRUNCATE_TEXT_BYTES]

    full_text = prepend_title(title, body).strip()

    if not full_text:
        metrics = analyze_text("")
        return ContentFeatures(
            post_id=post_id, content=body, title=title,
            source_id=source_id, source_type=source_type, post_date=post_date,
            text_length=metrics["text_length"], word_count=metrics["word_count"],
            reading_time=metrics["reading_time_min"], complexity=metrics["complexity"],
            is_short_form=metrics["is_short_form"], is_long_form=metrics["is_long_form"],
            processed_at=datetime.now(timezone.utc),
        )

    encoder = load_encoder()
    topic_pipe = load_topic_classifier()
    sentiment_pipe = load_sentiment_analyzer()

    nlp_text, was_truncated = extract_hmt(full_text, encoder.tokenizer, MAX_NLP_TOKENS)

    # Topic classification (torch_topic_classifier.py)
    with torch.no_grad():
        topic_result = topic_pipe(
            nlp_text,
            candidate_labels=TOPICS,
            hypothesis_template=HYPOTHESIS_TEMPLATE,
            multi_label=False,
            truncation=True,
            max_length=512,
        )
    labels = topic_result["labels"][:TOPIC_CLASSIFIER_TOP_K]
    scores = topic_result["scores"][:TOPIC_CLASSIFIER_TOP_K]
    topics_top = [(l, round(float(s), 4)) for l, s in zip(labels, scores)]

    # Sentiment (torch_sentiment_analyzer.py)
    with torch.no_grad():
        sent_res = sentiment_pipe(nlp_text, truncation=True, max_length=512)
    sent_label = str(sent_res[0]["label"]).upper()
    if sent_label not in {"POSITIVE", "NEGATIVE", "NEUTRAL"}:
        sent_label = "NEUTRAL"
    sent_score = round(float(sent_res[0]["score"]), 4)

    # NER (spaCy — на полном body, не на nlp_text)
    entities = extract_entities(body)

    # Embedding (L2-normalized)
    with torch.no_grad():
        emb = encoder.encode(
            [nlp_text], batch_size=1, convert_to_numpy=True, normalize_embeddings=True
        )[0]
    embedding = emb.tolist()

    # Text metrics (на полном body)
    metrics = analyze_text(body)

    return ContentFeatures(
        post_id=post_id,
        content=body,
        title=title,
        source_id=source_id,
        source_type=source_type,
        post_date=post_date,
        text_length=metrics["text_length"],
        word_count=metrics["word_count"],
        reading_time=metrics["reading_time_min"],
        complexity=metrics["complexity"],
        is_short_form=metrics["is_short_form"],
        is_long_form=metrics["is_long_form"],
        topic_1=topics_top[0][0] if len(topics_top) > 0 else None,
        topic_1_score=topics_top[0][1] if len(topics_top) > 0 else None,
        topic_2=topics_top[1][0] if len(topics_top) > 1 else None,
        topic_2_score=topics_top[1][1] if len(topics_top) > 1 else None,
        topic_3=topics_top[2][0] if len(topics_top) > 2 else None,
        topic_3_score=topics_top[2][1] if len(topics_top) > 2 else None,
        sentiment=sent_label,
        sentiment_score=sent_score,
        entities_persons=entities["persons"],
        entities_organizations=entities["organizations"],
        entities_locations=entities["locations"],
        embedding=embedding,
        processed_at=datetime.now(timezone.utc),
    )


def process_content_batch(raw_posts: list[dict], cache_key: str | None = None) -> list[ContentFeatures]:
    """Прогоняем батч постов через весь pipeline. Кеширует результат на диск."""
    if cache_key:
        cache_path = CACHE_DIR / f"features_{cache_key}.pkl"
        if cache_path.exists():
            print(f"Loading cached features from {cache_path.name}")
            with cache_path.open("rb") as f:
                return pickle.load(f)

    features: list[ContentFeatures] = []
    total = len(raw_posts)
    t0 = time.time()
    for i, p in enumerate(raw_posts, 1):
        print(f"  [{i:>3}/{total}] processing {p.get('title', '')[:60]!s}...", flush=True)
        features.append(process_single_post(p))
    dt = time.time() - t0
    print(f"Processed {total} posts in {dt:.1f}s ({dt / max(1, total):.2f}s/post)")

    if cache_key:
        with cache_path.open("wb") as f:
            pickle.dump(features, f)
        print(f"Cached to {cache_path.name}")

    return features


print("Content processing pipeline ready")


Content processing pipeline ready


## 10. Signal Classifier (`domain/services/signal_classifier.py`)

Переводит `UserInteraction` → `Signal` c нужным весом. Логика CLOSE проверяется строго по порядку
(fast → full → half → other). OPEN без парного CLOSE в пределах 5 мин → weight 0.


In [10]:
class SignalClassifier:
    def __init__(self, signal_weights: dict) -> None:
        self._w = signal_weights

    def classify_batch(self, interactions: list[UserInteraction]) -> list[Signal]:
        close_by_key: dict[tuple, list[UserInteraction]] = {}
        for i in interactions:
            if i.event_type == EventType.CLOSE:
                close_by_key.setdefault((i.user_id, i.post_id), []).append(i)
        signals = []
        for i in interactions:
            sig = self._classify_single(i, close_by_key)
            if sig is not None:
                signals.append(sig)
        return signals

    def _classify_single(self, i: UserInteraction, close_by_key) -> Signal | None:
        et = i.event_type
        if et == EventType.IMPRESSION:
            thr = self._w["impression_read"]["threshold_duration_ms"]
            w = (
                self._w["impression_read"]["weight"]
                if i.duration_ms is not None and i.duration_ms >= thr
                else self._w["impression_skip"]["weight"]
            )
            return Signal(i.post_id, i.user_id, w, EventType.IMPRESSION)
        if et == EventType.LIKE:
            return Signal(i.post_id, i.user_id, self._w["like"]["weight"], EventType.LIKE)
        if et == EventType.DISLIKE:
            return Signal(i.post_id, i.user_id, self._w["dislike"]["weight"], EventType.DISLIKE)
        if et == EventType.BOOKMARK:
            return Signal(i.post_id, i.user_id, self._w["bookmark"]["weight"], EventType.BOOKMARK)
        if et == EventType.CLOSE:
            return self._classify_close(i)
        if et == EventType.OPEN:
            return self._classify_open(i, close_by_key)
        return None

    def _classify_close(self, i: UserInteraction) -> Signal:
        dur = i.duration_ms or 0
        scroll = i.max_scroll_pct or 0.0
        fast_thr = self._w["close_fast"]["threshold_duration_ms"]
        if dur < fast_thr:
            w = self._w["close_fast"]["weight"]
        else:
            full_s = self._w["close_full"]["threshold_scroll_pct"]
            full_d = self._w["close_full"]["threshold_duration_ms"]
            half_s = self._w["close_half"]["threshold_scroll_pct"]
            half_d = self._w["close_half"]["threshold_duration_ms"]
            if scroll >= full_s and dur >= full_d:
                w = self._w["close_full"]["weight"]
            elif scroll >= half_s and dur >= half_d:
                w = self._w["close_half"]["weight"]
            else:
                w = self._w["close_other"]["weight"]
        return Signal(i.post_id, i.user_id, w, EventType.CLOSE)

    def _classify_open(self, i: UserInteraction, close_by_key) -> Signal:
        timeout = self._w.get("orphan_timeout_minutes", 5) * 60
        closes = close_by_key.get((i.user_id, i.post_id), [])
        paired = any(abs((c.created_at - i.created_at).total_seconds()) <= timeout for c in closes)
        w = self._w["open"]["weight"] if paired else 0.0
        return Signal(i.post_id, i.user_id, w, EventType.OPEN)


print("SignalClassifier ready")


SignalClassifier ready


## 11. Profile Updater (`domain/services/profile_updater.py`)

EMA-обновление профиля из батча сигналов:
- `topic_vector += lr * weight * topic_score` → L1-normalize
- `embedding = (1-α)*old + α*post.embedding`, α = lr * |weight| → L2-normalize (только позитивные сигналы)
- `sentiment_prefs += lr * weight * 0.5` → L1-normalize
- `format_prefs` EMA (только позитивные)
- `entity_interests` upsert для сигналов с weight ≥ 0.4


In [11]:
def _l2_normalize(vec: list[float]) -> list[float]:
    norm = math.sqrt(sum(x * x for x in vec))
    if norm == 0.0:
        return vec
    return [x / norm for x in vec]


@dataclass
class EntityInterest:
    user_id: uuid.UUID
    entity_type: str
    entity_name: str
    weight: float
    last_seen: datetime


class ProfileUpdater:
    def apply_batch(
        self,
        profile: UserProfile,
        signals: list[Signal],
        content_map: dict[uuid.UUID, ContentFeatures],
        config: dict,
    ) -> tuple[UserProfile, list[EntityInterest]]:
        lr = config.get("learning_rate", 0.08)
        entity_min = config.get("entity_min_weight", 0.4)
        entity_max = config.get("entity_max_per_post", 5)

        current = profile
        entity_changes: list[EntityInterest] = []

        for sig in signals:
            content = content_map.get(sig.post_id)
            if content is None:
                continue

            # topics
            new_tv = current.topic_vector.apply_update(lr, sig.weight, content.topic_dict())
            current = current.copy_with(topic_vector=new_tv)

            # embedding (positive only)
            if sig.is_positive and content.embedding is not None:
                if current.embedding is None:
                    new_emb = list(content.embedding)
                else:
                    alpha = lr * abs(sig.weight)
                    new_emb = [
                        (1 - alpha) * p + alpha * c
                        for p, c in zip(current.embedding, content.embedding)
                    ]
                    new_emb = _l2_normalize(new_emb)
                current = current.copy_with(embedding=new_emb)

            # sentiment prefs
            if content.sentiment is not None:
                new_sp = current.sentiment_prefs.apply_update(content.sentiment, lr, sig.weight)
                current = current.copy_with(sentiment_prefs=new_sp)

            # format prefs (positive only)
            if sig.is_positive and content.word_count is not None and content.complexity is not None:
                new_fp = current.format_prefs.apply_ema_update(
                    lr, float(content.word_count), content.complexity
                )
                current = current.copy_with(format_prefs=new_fp)

            # entity interests (strong signals)
            if sig.weight >= entity_min:
                for e in content.all_entities()[:entity_max]:
                    entity_changes.append(
                        EntityInterest(
                            user_id=sig.user_id,
                            entity_type="person",
                            entity_name=e,
                            weight=1.0,
                            last_seen=datetime.now(timezone.utc),
                        )
                    )

        updated = current.copy_with(
            interaction_count=current.interaction_count + len(signals),
            last_updated=datetime.now(timezone.utc),
        )
        return updated, entity_changes


print("ProfileUpdater ready")


ProfileUpdater ready


## 12. Onboarding Service (`domain/services/onboarding_service.py`)

Создаёт профиль из выбранных на онбординге тем (3-5 штук).
`topic_vector` = равная доля на выбранные темы + `baseline_weight` (0.01) на остальные.


In [12]:
def create_profile_from_onboarding(
    user_id: uuid.UUID,
    chosen_topics: list[str],
    config: dict = ONBOARDING_PARAMS,
) -> UserProfile:
    min_t = config.get("min_topics", 3)
    max_t = config.get("max_topics", 5)
    baseline = config.get("baseline_weight", 0.01)

    if len(chosen_topics) < min_t:
        raise ValueError(f"At least {min_t} topics required, got {len(chosen_topics)}")
    if len(chosen_topics) > max_t:
        raise ValueError(f"At most {max_t} topics allowed, got {len(chosen_topics)}")

    valid = set(TOPICS)
    for t in chosen_topics:
        if t not in valid:
            raise ValueError(f"Invalid topic '{t}'. Must be one of {TOPICS}")

    tv = TopicVector.create_from_selected(chosen_topics, baseline_weight=baseline)
    sp = SentimentPrefs.create_uniform()
    fp = FormatPrefs.create_default()
    now = datetime.now(timezone.utc)
    # cold_start=False matches prod OnboardUserUseCase (sets cold_start=False after onboarding)
    return UserProfile(
        user_id=user_id,
        topic_vector=tv,
        sentiment_prefs=sp,
        format_prefs=fp,
        interaction_count=0,
        created_at=now,
        last_updated=now,
        embedding=None,
        cold_start=False,
    )


print("Onboarding service ready")


Onboarding service ready


## 13. Scorer — 6-component formula (`domain/services/scorer.py`)

```
score = w_topic * topic_match        # sum(profile.topics[t] * post.topic_score[t])
      + w_emb * embedding_sim        # max(0, cos(profile.emb, post.emb))
      + w_ent * entity_match         # min(matches / 3, 1.0)
      + w_sent * sentiment_match     # profile.sentiment_prefs[post.sentiment]
      + w_fresh * freshness          # exp(-ln(2) * hours / halflife)
      + w_fmt * format_match         # length_match * complexity_match
```

**Cold start redistribution**: если `profile.embedding is None` или нет entity_interests — веса
«мёртвых» компонент пропорционально распределяются по живым.


In [13]:
class Scorer:
    def score_batch(
        self,
        candidates: list[tuple[ContentFeatures, set[str]]],
        profile: UserProfile,
        config: dict,
    ) -> list[tuple[uuid.UUID, float]]:
        weights = config.get("scoring_weights", SCORING_WEIGHTS)
        halflife = config.get("freshness_halflife_hours", 48)
        out: list[tuple[uuid.UUID, float]] = []
        for content, ents in candidates:
            s = self._compute_score(profile, content, ents, weights, halflife)
            out.append((content.post_id, s))
        out.sort(key=lambda x: x[1], reverse=True)
        return out

    def _compute_score(
        self,
        profile: UserProfile,
        content: ContentFeatures,
        entity_interests: set[str],
        w: dict,
        halflife: float,
    ) -> float:
        topic = self.topic_match(profile, content)
        emb = self.embedding_sim(profile, content)
        ent = self.entity_match(content, entity_interests)
        sent = self.sentiment_match(profile, content)
        fresh = self.freshness(content, halflife)
        fmt = self.format_match(profile, content)

        emb_dead = profile.embedding is None
        ent_dead = len(entity_interests) == 0 and len(content.all_entities()) == 0

        dead_weight = 0.0
        if emb_dead:
            dead_weight += w.get("embedding_sim", 0.25)
        if ent_dead:
            dead_weight += w.get("entity_match", 0.15)

        if dead_weight > 0.0:
            active: dict[str, tuple[float, float]] = {
                "topic_match": (topic, w.get("topic_match", 0.30)),
                "sentiment_match": (sent, w.get("sentiment_match", 0.05)),
                "freshness": (fresh, w.get("freshness", 0.15)),
                "format_match": (fmt, w.get("format_match", 0.10)),
            }
            if not emb_dead:
                active["embedding_sim"] = (emb, w.get("embedding_sim", 0.25))
            if not ent_dead:
                active["entity_match"] = (ent, w.get("entity_match", 0.15))
            total_w = sum(cw for _, cw in active.values())
            if total_w == 0.0:
                return 0.0
            score = 0.0
            for val, cw in active.values():
                redistributed = cw + dead_weight * (cw / total_w)
                score += redistributed * val
            return score

        return (
            w.get("topic_match", 0.30) * topic
            + w.get("embedding_sim", 0.25) * emb
            + w.get("entity_match", 0.15) * ent
            + w.get("sentiment_match", 0.05) * sent
            + w.get("freshness", 0.15) * fresh
            + w.get("format_match", 0.10) * fmt
        )

    def topic_match(self, profile: UserProfile, content: ContentFeatures) -> float:
        topics = content.topic_dict()
        m = sum(profile.topic_vector.get(t) * s for t, s in topics.items())
        return min(max(m, 0.0), 1.0)

    def embedding_sim(self, profile: UserProfile, content: ContentFeatures) -> float:
        if profile.embedding is None or content.embedding is None:
            return 0.0
        dot = sum(p * c for p, c in zip(profile.embedding, content.embedding))
        return max(0.0, min(dot, 1.0))

    def entity_match(self, content: ContentFeatures, ents: set[str]) -> float:
        all_e = content.all_entities()
        if not all_e:
            return 0.0
        matches = sum(1 for e in all_e if e in ents)
        return min(matches / 3.0, 1.0)

    def sentiment_match(self, profile: UserProfile, content: ContentFeatures) -> float:
        if content.sentiment is None:
            return 0.0
        return profile.sentiment_prefs.get(content.sentiment)

    def freshness(self, content: ContentFeatures, halflife_hours: float) -> float:
        now = datetime.now(timezone.utc)
        pd = content.post_date
        if pd.tzinfo is None:
            pd = pd.replace(tzinfo=timezone.utc)
        hours = max(0.0, (now - pd).total_seconds() / 3600.0)
        return math.exp(-0.693 * hours / halflife_hours)

    def format_match(self, profile: UserProfile, content: ContentFeatures) -> float:
        if content.word_count is None or content.complexity is None:
            return 0.0
        length_match = max(0.0, 1.0 - abs(content.word_count - profile.format_prefs.avg_length) / 500)
        complexity_match = max(0.0, 1.0 - abs(content.complexity - profile.format_prefs.avg_complexity))
        return length_match * complexity_match


print("Scorer ready")


Scorer ready


## 14. Diversity Filter (`domain/services/diversity_filter.py`)

Три ограничения:
1. **topic streak** — не более `max_topic_streak` (=3) подряд одной темы.
2. **topic global cap** — не более `max_topic_ratio` (=0.40) от feed на одну тему.
3. **related spacing** — пары из `similarities.RELATED` разводятся минимум на `related_min_gap` (=3).


In [14]:
class DiversityFilter:
    def filter(
        self,
        candidates: list[tuple[ContentFeatures, float]],
        feed_size: int,
        config: dict,
        related_map: dict[uuid.UUID, set[uuid.UUID]] | None = None,
    ) -> list[tuple[uuid.UUID, float]]:
        max_streak = config.get("max_topic_streak", 3)
        max_ratio = config.get("max_topic_ratio", 0.40)
        related_min_gap = config.get("related_min_gap", 3)
        max_per_topic = feed_size * max_ratio

        feed: list[tuple[uuid.UUID, float]] = []
        streak: dict[str, int] = {}
        count: dict[str, int] = {}
        last_topic: str | None = None
        pos_of: dict[uuid.UUID, int] = {}

        for content, score in candidates:
            topic = content.topic_1 or ""
            streak.setdefault(topic, 0)
            count.setdefault(topic, 0)

            if streak[topic] >= max_streak:
                continue
            if count[topic] >= max_per_topic:
                continue

            if related_map:
                rels = related_map.get(content.post_id, set())
                conflict = False
                for rid in rels:
                    if rid in pos_of and (len(feed) - pos_of[rid]) < related_min_gap:
                        conflict = True
                        break
                if conflict:
                    continue

            pos_of[content.post_id] = len(feed)
            feed.append((content.post_id, score))
            count[topic] = count.get(topic, 0) + 1

            if topic == last_topic:
                streak[topic] += 1
            else:
                streak[topic] = 1
                for other in streak:
                    if other != topic:
                        streak[other] = 0
            last_topic = topic

            if len(feed) >= feed_size:
                break
        return feed


print("DiversityFilter ready")


DiversityFilter ready


## 15. Feed Generation Orchestrator (`application/use_cases/generate_feed.py`)

Повторяет `GenerateFeedUseCase.execute` для NORMAL-режима:
1. Process unprocessed interactions inline → update profile.
2. Load profile.
3. Candidate retrieval: freshness top-K + embedding top-K → dedupe.
4. Dedup clustering по `similarities.EXACT/DUPLICATE` (в ноутбуке — stub, оставляем всех кандидатов).
5. 6-component scorer → sort DESC.
6. Diversity filter.
7. Exclude already-recommended posts (history).
8. Save to history + return ordered list.


In [15]:
# Глобальное состояние playground — эмулирует таблицы БД.
SCORER = Scorer()
DIVERSITY = DiversityFilter()
SIGNAL_CLASSIFIER = SignalClassifier(SIGNAL_WEIGHTS)
PROFILE_UPDATER = ProfileUpdater()


class Playground:
    """In-memory эмулятор БД + репозиториев rec-system.

    Содержит:
    - features_by_id: posts_features (post_id → ContentFeatures)
    - profiles: rec_profiles (user_id → UserProfile)
    - entity_interests: rec_entity_interests (user_id → set[str])
    - interactions: user_interactions (list)
    - recommendation_history: (user_id, content_id) set
    - related_map, dedup_clusters: similarities snapshot (пусто по умолчанию)
    """

    def __init__(self, features: list[ContentFeatures]) -> None:
        self.features_by_id: dict[uuid.UUID, ContentFeatures] = {f.post_id: f for f in features}
        self.profiles: dict[uuid.UUID, UserProfile] = {}
        self.entity_interests: dict[uuid.UUID, set[str]] = defaultdict(set)
        self.interactions: list[UserInteraction] = []
        self.rec_history: set[tuple[uuid.UUID, uuid.UUID]] = set()
        self.related_map: dict[uuid.UUID, set[uuid.UUID]] = {}
        self.dedup_clusters: dict[uuid.UUID, uuid.UUID] = {}

    # --- Candidate retrieval (pg_content_repository.py) ---
    def get_candidates_by_freshness(
        self, max_age_hours: int, limit: int, exclude_source_ids: list[uuid.UUID],
    ) -> list[ContentFeatures]:
        now = datetime.now(timezone.utc)
        cutoff = now - timedelta(hours=max_age_hours)
        excl = set(exclude_source_ids)
        out = [
            f for f in self.features_by_id.values()
            if (f.processed_at or f.post_date) >= cutoff and f.source_id not in excl
        ]
        out.sort(key=lambda f: (f.processed_at or f.post_date), reverse=True)
        return out[:limit]

    def get_candidates_by_embedding(
        self, embedding: list[float], limit: int, exclude_source_ids: list[uuid.UUID],
    ) -> list[ContentFeatures]:
        excl = set(exclude_source_ids)

        def cos_dist(f: ContentFeatures) -> float:
            if f.embedding is None:
                return 2.0
            dot = sum(p * c for p, c in zip(embedding, f.embedding))
            return 1.0 - max(-1.0, min(1.0, dot))

        scored = [
            (f, cos_dist(f)) for f in self.features_by_id.values()
            if f.embedding is not None and f.source_id not in excl
        ]
        scored.sort(key=lambda x: x[1])
        return [f for f, _ in scored[:limit]]

    def get_published_ids(self, post_ids: list[uuid.UUID]) -> dict[uuid.UUID, uuid.UUID]:
        # В production post_id (raw_content) и published_content.id — разные.
        # В ноутбуке упрощаем: post_id == published_id.
        return {pid: pid for pid in post_ids if pid in self.features_by_id}


def generate_feed(
    pg: Playground,
    user_id: uuid.UUID,
    feed_size: int = 10,
    exclude_source_ids: list[uuid.UUID] | None = None,
    config: dict | None = None,
    respect_history: bool = True,
) -> dict:
    """Полный pipeline GenerateFeedUseCase.execute (NORMAL mode)."""
    cfg = {**FEED_CONFIG, **(config or {})}
    t0 = time.time()

    # 0. Process unprocessed interactions inline
    unprocessed = [i for i in pg.interactions if i.user_id == user_id and not i.processed]
    events_processed = 0
    if unprocessed and user_id in pg.profiles:
        signals = SIGNAL_CLASSIFIER.classify_batch(unprocessed)
        profile = pg.profiles[user_id]
        updated, ent_changes = PROFILE_UPDATER.apply_batch(
            profile, signals, pg.features_by_id, cfg,
        )
        pg.profiles[user_id] = updated
        for ec in ent_changes:
            pg.entity_interests[ec.user_id].add(ec.entity_name)
        for i in unprocessed:
            i.processed = True
        events_processed = len(unprocessed)

    profile = pg.profiles.get(user_id)
    if profile is None:
        return {"feed": [], "meta": {"error": "profile_not_found"}}

    # 1. Candidate retrieval (two-stage: freshness + embedding)
    exclude = exclude_source_ids or []
    max_age_hours = cfg.get("max_age_hours", 48)
    fresh_limit = cfg.get("freshness_candidate_limit", 500)
    emb_limit = cfg.get("embedding_candidate_limit", 500)

    fresh = pg.get_candidates_by_freshness(max_age_hours, fresh_limit, exclude)
    emb: list[ContentFeatures] = []
    if profile.embedding is not None:
        emb = pg.get_candidates_by_embedding(profile.embedding, emb_limit, exclude)

    seen: set[uuid.UUID] = set()
    candidates: list[ContentFeatures] = []
    for c in fresh + emb:
        if c.post_id in seen:
            continue
        seen.add(c.post_id)
        candidates.append(c)

    if not candidates:
        return {"feed": [], "meta": {"total": 0, "events_processed": events_processed}}

    # 2. Dedup clustering — в ноутбуке stub; при наличии pg.dedup_clusters — применяем Union-Find.
    if pg.dedup_clusters:
        groups = defaultdict(list)
        for c in candidates:
            groups[pg.dedup_clusters.get(c.post_id, c.post_id)].append(c)
        candidates = [max(g, key=lambda c: c.post_date) for g in groups.values()]

    # 3. Scoring
    ent_set = pg.entity_interests.get(user_id, set())
    scoring_input = [(c, ent_set) for c in candidates]
    scored = SCORER.score_batch(scoring_input, profile, cfg)
    content_map = {c.post_id: c for c in candidates}

    # 4. Diversity filter
    div_input = [(content_map[pid], s) for pid, s in scored if pid in content_map]
    related = pg.related_map if cfg.get("enable_related_spacing", True) else None
    filtered = DIVERSITY.filter(div_input, feed_size * 5, cfg, related_map=related)

    # 5. History exclusion
    if respect_history:
        filtered = [(pid, s) for pid, s in filtered if (user_id, pid) not in pg.rec_history]
    filtered = filtered[:feed_size]

    # 6. Save to history
    for pid, _ in filtered:
        pg.rec_history.add((user_id, pid))

    # 7. Build response
    published = pg.get_published_ids([pid for pid, _ in filtered])
    feed: list[dict] = []
    for pid, score in filtered:
        pub = published.get(pid)
        if pub is None:
            continue
        c = pg.features_by_id[pid]
        feed.append({
            "post_id": str(pub),
            "score": round(score, 4),
            "title": c.title,
            "topic_1": c.topic_1, "topic_1_score": c.topic_1_score,
            "topic_2": c.topic_2, "topic_3": c.topic_3,
            "sentiment": c.sentiment,
            "word_count": c.word_count,
            "post_date": c.post_date.isoformat() if c.post_date else None,
        })

    return {
        "feed": feed,
        "meta": {
            "total": len(feed),
            "candidates_scored": len(candidates),
            "events_processed": events_processed,
            "generation_time_ms": int((time.time() - t0) * 1000),
        },
    }


print("generate_feed orchestrator ready")


generate_feed orchestrator ready


---

# 🚀 Run the playground

Ниже — сценарий использования. Подставьте свой CSV в `CSV_PATH`.


## Step 1. Load raw content CSV

Ожидаемая схема CSV (колонки):
- `id` — UUID (любой уникальный идентификатор)
- `title` — заголовок
- `body` — тело поста
- `source_id` — UUID источника
- `source_type` — `TELEGRAM` / `RSS` / …
- `post_date` — ISO 8601 datetime (с таймзоной)


In [16]:
import hashlib

CSV_PATH = PROJECT_ROOT / "rec_playground_sample.csv"
assert CSV_PATH.exists(), f"CSV не найден: {CSV_PATH}"

raw_df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(raw_df)} rows from {CSV_PATH.name}")
print(raw_df[["id", "title", "source_type"]].head(5).to_string(index=False))

raw_posts: list[dict] = raw_df.to_dict(orient="records")
# Кеш-ключ: хеш содержимого файла (не протухает, если CSV не менялся)
CSV_HASH = hashlib.md5(CSV_PATH.read_bytes()).hexdigest()[:12]
print(f"\nCSV hash: {CSV_HASH}")


Loaded 12 rows from rec_playground_sample.csv
                                  id                               title source_type
11111111-1111-1111-1111-000000000001              Новый смартфон Яндекса    TELEGRAM
11111111-1111-1111-1111-000000000002 Обновление ChatGPT и конкуренция ИИ    TELEGRAM
11111111-1111-1111-1111-000000000003             Квантовый процессор IBM    TELEGRAM
11111111-1111-1111-1111-000000000004       Финал Кубка России по футболу    TELEGRAM
11111111-1111-1111-1111-000000000005        Олимпийская сборная по лыжам    TELEGRAM

CSV hash: 820bd19b3f20


## Step 2. Run NLP pipeline (with cache)

Первый прогон на 12 постах ≈ 30-60 секунд на CPU. Повторные запуски читают из `.playground_cache/`.


In [17]:
features = process_content_batch(raw_posts, cache_key=CSV_HASH)

# Краткий отчёт
feat_df = pd.DataFrame([
    {
        "post_id": str(f.post_id)[:8],
        "title": (f.title or "")[:40],
        "topic_1": f.topic_1, "t1_score": f.topic_1_score,
        "topic_2": f.topic_2,
        "sentiment": f.sentiment,
        "word_count": f.word_count,
        "complexity": f.complexity,
        "emb_dim": len(f.embedding) if f.embedding else 0,
        "persons": len(f.entities_persons), "orgs": len(f.entities_organizations), "locs": len(f.entities_locations),
    }
    for f in features
])
print(feat_df.to_string(index=False))


  [  1/12] processing Новый смартфон Яндекса...


  [  2/12] processing Обновление ChatGPT и конкуренция ИИ...


  [  3/12] processing Квантовый процессор IBM...


  [  4/12] processing Финал Кубка России по футболу...


  [  5/12] processing Олимпийская сборная по лыжам...


...

  [ 10/12] processing Открытие экзопланеты телескопом Джеймс Уэбб...


  [ 11/12] processing Новый ускоритель частиц в ЦЕРНе...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [ 12/12] processing Парламентские выборы в Европе...


Processed 12 posts in 2.6s (0.21s/post)
Cached to features_820bd19b3f20.pkl
 post_id                                    title      topic_1  t1_score      topic_2 sentiment  word_count  complexity  emb_dim  persons  orgs  locs
11111111                   Новый смартфон Яндекса   технологии    0.5756       бизнес   NEUTRAL          48      0.8387      312        2     0     1
11111111      Обновление ChatGPT и конкуренция ИИ       бизнес    0.4747   технологии   NEUTRAL          45      0.6780      312        0     4     1
11111111                  Квантовый процессор IBM        наука    0.4848   технологии   NEUTRAL          45      0.8116      312        0     1     3
11111111            Финал Кубка России по футболу        спорт    0.9507     культура   NEUTRAL          45      0.3781      312        1     1     2
11111111             Олимпийская сборная по лыжам        спорт    0.7357    транспорт  POSITIVE          39      0.7128      312        0     0     2
11111111                

## Step 3. Initialise `Playground` (in-memory DB)

In [18]:
pg = Playground(features)
print(f"Playground created with {len(pg.features_by_id)} posts.")
print("Topic distribution (by topic_1):")
topic_counts = pd.Series([f.topic_1 for f in features]).value_counts()
print(topic_counts.to_string())


Playground created with 12 posts.
Topic distribution (by topic_1):
наука           4
спорт           3
технологии      1
бизнес          1
происшествия    1
здоровье        1
политика        1


## Step 4. Define N personas and onboard them

Каждая персона — имитация реального пользователя со своими интересами.
`onboarding_topics` = темы, которые пользователь «выбрал» на онбординге (3-5 шт.).
`persona_preferred` / `persona_avoid` используются в симуляторе взаимодействий (ниже).


In [19]:
# === Персоны ===
PERSONAS: list[dict] = [
    {
        "user_id": uuid.UUID("aaaa1111-0000-0000-0000-000000000001"),
        "name": "Tech & Science Enthusiast",
        "onboarding_topics": ["технологии", "наука", "образование"],
        "persona_preferred": {"технологии", "наука"},
        "persona_avoid": {"спорт", "культура"},
        "read_probability": 0.8,     # при совпадении темы — высокий шанс вчитаться
        "like_probability": 0.5,     # из прочитанных — половина лайков
        "bookmark_probability": 0.3,
        "dislike_probability_offtopic": 0.4,  # дизлайк чужой темы
    },
    {
        "user_id": uuid.UUID("aaaa2222-0000-0000-0000-000000000002"),
        "name": "Sports Fan",
        "onboarding_topics": ["спорт", "здоровье", "развлечения"],
        "persona_preferred": {"спорт", "здоровье"},
        "persona_avoid": {"политика", "международные новости"},
        "read_probability": 0.75,
        "like_probability": 0.4,
        "bookmark_probability": 0.2,
        "dislike_probability_offtopic": 0.5,
    },
    {
        "user_id": uuid.UUID("aaaa3333-0000-0000-0000-000000000003"),
        "name": "Health Conscious",
        "onboarding_topics": ["здоровье", "наука", "природа"],
        "persona_preferred": {"здоровье", "наука"},
        "persona_avoid": {"спорт", "криминал"},
        "read_probability": 0.85,
        "like_probability": 0.6,
        "bookmark_probability": 0.4,
        "dislike_probability_offtopic": 0.3,
    },
]

# === Onboarding ===
for p in PERSONAS:
    prof = create_profile_from_onboarding(p["user_id"], p["onboarding_topics"])
    pg.profiles[p["user_id"]] = prof

# Печатаем начальные topic_vector
print(f"{len(pg.profiles)} profiles onboarded.\n")
for p in PERSONAS:
    prof = pg.profiles[p["user_id"]]
    top5 = sorted(prof.topic_vector.weights.items(), key=lambda kv: -kv[1])[:5]
    print(f"[{p['name']}] onboarded with {p['onboarding_topics']}")
    print(f"  top-5 topic weights: {top5}")
    print(f"  embedding: {'None (cold-start)' if prof.embedding is None else 'set'}")
    print()


3 profiles onboarded.

[Tech & Science Enthusiast] onboarded with ['технологии', 'наука', 'образование']
  top-5 topic weights: [('технологии', 0.31746031746031744), ('наука', 0.31746031746031744), ('образование', 0.31746031746031744), ('политика', 0.0031746031746031746), ('экономика', 0.0031746031746031746)]
  embedding: None (cold-start)

[Sports Fan] onboarded with ['спорт', 'здоровье', 'развлечения']
  top-5 topic weights: [('спорт', 0.31746031746031744), ('здоровье', 0.31746031746031744), ('развлечения', 0.31746031746031744), ('политика', 0.0031746031746031746), ('экономика', 0.0031746031746031746)]
  embedding: None (cold-start)

[Health Conscious] onboarded with ['здоровье', 'наука', 'природа']
  top-5 topic weights: [('наука', 0.31746031746031744), ('здоровье', 0.31746031746031744), ('природа', 0.31746031746031744), ('политика', 0.0031746031746031746), ('экономика', 0.0031746031746031746)]
  embedding: None (cold-start)



## Step 5. Generate initial feed for each persona

На этом шаге `profile.embedding is None` → cold_start redistribution в Scorer.
Вес `embedding_sim` (0.25) и `entity_match` (0.15) перераспределяются на активные компоненты.


In [20]:
FEED_SIZE = 8  # маленький feed — всего 12 постов

def feed_summary(feed: dict, limit: int = None) -> pd.DataFrame:
    rows = feed["feed"][:limit] if limit else feed["feed"]
    return pd.DataFrame([
        {
            "rank": i + 1,
            "title": (r["title"] or "")[:45],
            "score": r["score"],
            "topic_1": r["topic_1"],
            "t1_score": r["topic_1_score"],
            "sentiment": r["sentiment"],
        }
        for i, r in enumerate(rows)
    ])


INITIAL_FEEDS: dict[uuid.UUID, dict] = {}
for p in PERSONAS:
    # Обнуляем rec_history перед генерацией (чтобы инициальный feed был полный)
    pg.rec_history = {(u, c) for (u, c) in pg.rec_history if u != p["user_id"]}
    feed = generate_feed(pg, p["user_id"], feed_size=FEED_SIZE)
    INITIAL_FEEDS[p["user_id"]] = feed
    print(f"\n=== Initial feed for [{p['name']}] ===")
    print(f"meta: {feed['meta']}")
    print(feed_summary(feed).to_string(index=False))



=== Initial feed for [Tech & Science Enthusiast] ===
meta: {'total': 8, 'candidates_scored': 12, 'events_processed': 0, 'generation_time_ms': 0}
 rank                                       title  score    topic_1  t1_score sentiment
    1 Открытие экзопланеты телескопом Джеймс Уэбб 0.3757      наука    0.4901   NEUTRAL
    2                     Квантовый процессор IBM 0.3702      наука    0.4848   NEUTRAL
    3               Рекомендации по здоровому сну 0.3487   здоровье    0.7857  POSITIVE
    4                    Прорыв в лечении диабета 0.3436      наука    0.6832   NEUTRAL
    5                      Новый смартфон Яндекса 0.3260 технологии    0.5756   NEUTRAL
    6             Новый ускоритель частиц в ЦЕРНе 0.3186      наука    0.4546   NEUTRAL
    7         Обновление ChatGPT и конкуренция ИИ 0.3078     бизнес    0.4747   NEUTRAL
    8               Финал Кубка России по футболу 0.2939      спорт    0.9507   NEUTRAL

=== Initial feed for [Sports Fan] ===
meta: {'total': 8, 'can

## Step 6. Simulate user sessions (frontend-like interactions)

Эмулируем поведение фронтенда:
- для каждого поста из фида: `IMPRESSION` (всегда)
- если тема совпадает с предпочтениями → `OPEN` → `CLOSE` с большим scroll и duration → возможно `LIKE` / `BOOKMARK`
- если тема в `persona_avoid` → `CLOSE` с низким duration → возможно `DISLIKE`

Все events получают UUID, сохраняются в `user_interactions`, processed=False.
`generate_feed` при следующем вызове прогонит их через `SignalClassifier` → `ProfileUpdater`.


In [21]:
_interaction_counter = 0


def _new_interaction_id() -> int:
    global _interaction_counter
    _interaction_counter += 1
    return _interaction_counter


def simulate_session(
    pg: Playground,
    user_id: uuid.UUID,
    feed: dict,
    persona: dict,
    rng: random.Random | None = None,
) -> list[UserInteraction]:
    """Генерирует события фронтенда для feed'а и сохраняет в pg.interactions.

    Логика:
    1. Каждый элемент feed получает IMPRESSION (как «показ карточки»).
    2. При совпадении темы с preferred → OPEN + CLOSE (долгое, scroll высокий).
    3. С вероятностью like/bookmark — соответствующее событие.
    4. При совпадении темы с avoid → CLOSE (быстрое) ± DISLIKE.
    """
    rng = rng or random.Random(SEED)
    now = datetime.now(timezone.utc)
    preferred = persona["persona_preferred"]
    avoid = persona["persona_avoid"]
    read_p = persona["read_probability"]
    like_p = persona["like_probability"]
    bookmark_p = persona["bookmark_probability"]
    dislike_p = persona["dislike_probability_offtopic"]

    events: list[UserInteraction] = []

    def _add(event_type: str, post_id: uuid.UUID, *, duration_ms=None, scroll=None, max_scroll=None, t_offset=0) -> None:
        ev = UserInteraction(
            id=_new_interaction_id(),
            event_id=uuid.uuid4(),
            user_id=user_id,
            post_id=post_id,
            event_type=event_type,
            duration_ms=duration_ms,
            scroll_pct=scroll,
            max_scroll_pct=max_scroll,
            created_at=now + timedelta(seconds=t_offset),
            processed=False,
        )
        pg.interactions.append(ev)
        events.append(ev)

    for idx, item in enumerate(feed["feed"]):
        post_id = uuid.UUID(item["post_id"])
        topic = item["topic_1"]

        # 1. Всегда IMPRESSION — карточка показана.
        impression_duration = rng.randint(1500, 4500) if topic in preferred else rng.randint(500, 2500)
        _add(EventType.IMPRESSION, post_id, duration_ms=impression_duration, t_offset=idx * 10)

        in_pref = topic in preferred
        in_avoid = topic in avoid

        if in_pref and rng.random() < read_p:
            # открыл и прочитал
            _add(EventType.OPEN, post_id, t_offset=idx * 10 + 1)
            duration = rng.randint(15000, 40000)
            scroll = rng.uniform(0.75, 1.0)
            _add(EventType.CLOSE, post_id,
                 duration_ms=duration, scroll=scroll, max_scroll=scroll, t_offset=idx * 10 + 5)
            if rng.random() < like_p:
                _add(EventType.LIKE, post_id, t_offset=idx * 10 + 6)
            if rng.random() < bookmark_p:
                _add(EventType.BOOKMARK, post_id, t_offset=idx * 10 + 7)
        elif in_avoid:
            # быстро закрыл
            _add(EventType.OPEN, post_id, t_offset=idx * 10 + 1)
            duration = rng.randint(500, 2500)
            scroll = rng.uniform(0.05, 0.25)
            _add(EventType.CLOSE, post_id,
                 duration_ms=duration, scroll=scroll, max_scroll=scroll, t_offset=idx * 10 + 2)
            if rng.random() < dislike_p:
                _add(EventType.DISLIKE, post_id, t_offset=idx * 10 + 3)
        else:
            # нейтральная тема — половина на половину
            if rng.random() < 0.4:
                _add(EventType.OPEN, post_id, t_offset=idx * 10 + 1)
                duration = rng.randint(4000, 12000)
                scroll = rng.uniform(0.3, 0.6)
                _add(EventType.CLOSE, post_id,
                     duration_ms=duration, scroll=scroll, max_scroll=scroll, t_offset=idx * 10 + 4)
    return events


print("simulate_session ready")


simulate_session ready


## Step 7. Run journey: N sessions per user

Для каждой персоны:
1. Генерируем feed → сохраняем snapshot.
2. Симулируем сессию → events попадают в `pg.interactions`.
3. Следующий вызов `generate_feed` применит `SignalClassifier` + `ProfileUpdater` inline.
4. Повторяем N раз.

В финале видим, как изменилось topic_vector после каждой итерации.


In [22]:
N_SESSIONS = 3


def topic_vector_snapshot(profile: UserProfile, top_n: int = 5) -> list[tuple[str, float]]:
    return sorted(profile.topic_vector.weights.items(), key=lambda kv: -kv[1])[:top_n]


journeys: dict[uuid.UUID, list[dict]] = defaultdict(list)

for p in PERSONAS:
    user_id = p["user_id"]
    rng = random.Random(SEED + hash(p["name"]) % 1000)
    # Обнуляем историю рекомендаций для чистого эксперимента
    pg.rec_history = {(u, c) for (u, c) in pg.rec_history if u != user_id}
    # Сбрасываем взаимодействия этого юзера
    pg.interactions = [i for i in pg.interactions if i.user_id != user_id]
    # Сбрасываем профиль к онбордингу
    pg.profiles[user_id] = create_profile_from_onboarding(user_id, p["onboarding_topics"])

    for session_idx in range(N_SESSIONS):
        profile_before = pg.profiles[user_id]
        tv_before = topic_vector_snapshot(profile_before)

        feed = generate_feed(pg, user_id, feed_size=FEED_SIZE)
        events = simulate_session(pg, user_id, feed, p, rng=rng)

        journeys[user_id].append({
            "session": session_idx + 1,
            "feed": feed["feed"],
            "meta": feed["meta"],
            "events": [
                {"type": e.event_type, "post_id": str(e.post_id)[:8],
                 "duration_ms": e.duration_ms, "max_scroll_pct": e.max_scroll_pct}
                for e in events
            ],
            "tv_before": tv_before,
            "has_embedding_before": profile_before.embedding is not None,
            "interaction_count_before": profile_before.interaction_count,
        })

    # После последней сессии — насильно прогоняем накопленные events, чтобы отразить в профиле.
    # В prod это делает следующий вызов /recommendations или APScheduler job.
    _ = generate_feed(pg, user_id, feed_size=0)  # feed_size=0 → ничего не вернёт, но профиль обновит

    # Финальный snapshot профиля
    profile_final = pg.profiles[user_id]
    journeys[user_id].append({
        "session": "FINAL",
        "feed": [],
        "events": [],
        "tv_before": topic_vector_snapshot(profile_final),
        "has_embedding_before": profile_final.embedding is not None,
        "interaction_count_before": profile_final.interaction_count,
    })

print("Journey done for all personas:")
for p in PERSONAS:
    print(f"  [{p['name']}] — {N_SESSIONS} sessions, "
          f"{len([i for i in pg.interactions if i.user_id == p['user_id']])} events total, "
          f"final interaction_count = {pg.profiles[p['user_id']].interaction_count}")


Journey done for all personas:
  [Tech & Science Enthusiast] — 3 sessions, 34 events total, final interaction_count = 34
  [Sports Fan] — 3 sessions, 30 events total, final interaction_count = 30
  [Health Conscious] — 3 sessions, 41 events total, final interaction_count = 41


## Step 8. Journey report

Для каждого пользователя отчёт включает:
- feed по каждой сессии (с темой, sentiment, score)
- события, которые он совершил
- изменение topic_vector (топ-5) между сессиями
- что изменилось в embedding'е (появился / сместился)


In [23]:
def print_journey_report(user_id: uuid.UUID, persona: dict) -> None:
    print("=" * 90)
    print(f"JOURNEY REPORT — {persona['name']}")
    print(f"user_id: {user_id}")
    print(f"onboarding_topics: {persona['onboarding_topics']}")
    print(f"preferred: {sorted(persona['persona_preferred'])}")
    print(f"avoid:     {sorted(persona['persona_avoid'])}")
    print("=" * 90)

    for step in journeys[user_id]:
        s = step["session"]
        print(f"\n--- Session {s} ---")
        print(f"  profile.interaction_count = {step['interaction_count_before']}, "
              f"has_embedding = {step['has_embedding_before']}")
        print(f"  topic_vector (top-5) before:")
        for t, w in step["tv_before"]:
            mark = " *" if t in persona["persona_preferred"] else ("  ×" if t in persona["persona_avoid"] else "")
            print(f"    {t:28s} {w:.4f}{mark}")

        if step["feed"]:
            print(f"  feed (top {len(step['feed'])}):")
            for i, r in enumerate(step["feed"], 1):
                mark = "✓" if r["topic_1"] in persona["persona_preferred"] else (
                    "✗" if r["topic_1"] in persona["persona_avoid"] else " "
                )
                print(f"    {i}. {mark} [{r['topic_1']:18s} {r['topic_1_score']:.2f}] "
                      f"score={r['score']:.3f}  {r['sentiment']:8s}  "
                      f"{(r['title'] or '')[:45]}")

        if step["events"]:
            counts = defaultdict(int)
            for e in step["events"]:
                counts[e["type"]] += 1
            print(f"  events: {dict(counts)}")


for p in PERSONAS:
    print_journey_report(p["user_id"], p)
    print("\n")


JOURNEY REPORT — Tech & Science Enthusiast
user_id: aaaa1111-0000-0000-0000-000000000001
onboarding_topics: ['технологии', 'наука', 'образование']
preferred: ['наука', 'технологии']
avoid:     ['культура', 'спорт']

--- Session 1 ---
  profile.interaction_count = 0, has_embedding = False
  topic_vector (top-5) before:
    технологии                   0.3175 *
    наука                        0.3175 *
    образование                  0.3175
    политика                     0.0032
    экономика                    0.0032
  feed (top 8):
    1. ✓ [наука              0.49] score=0.376  NEUTRAL   Открытие экзопланеты телескопом Джеймс Уэбб
    2. ✓ [наука              0.48] score=0.370  NEUTRAL   Квантовый процессор IBM
    3.   [здоровье           0.79] score=0.349  POSITIVE  Рекомендации по здоровому сну
    4. ✓ [наука              0.68] score=0.344  NEUTRAL   Прорыв в лечении диабета
    5. ✓ [технологии         0.58] score=0.326  NEUTRAL   Новый смартфон Яндекса
    6. ✓ [наука         

## Step 9. Full metadata — what each user saw

Таблица со всеми постами из всех фидов каждого пользователя + полная мета-информация.
Полезно, если хочется посмотреть, какие конкретно фичи (topic_1/2/3, sentiment, entities)
повлияли на решение рекомендательной системы.


In [24]:
def full_journey_table(user_id: uuid.UUID) -> pd.DataFrame:
    rows = []
    for step in journeys[user_id]:
        if not step["feed"]:
            continue
        for rank, item in enumerate(step["feed"], 1):
            pid = uuid.UUID(item["post_id"])
            feat = pg.features_by_id[pid]
            rows.append({
                "session": step["session"],
                "rank": rank,
                "title": (feat.title or "")[:35],
                "score": item["score"],
                "topic_1": feat.topic_1,
                "t1_score": feat.topic_1_score,
                "topic_2": feat.topic_2,
                "topic_3": feat.topic_3,
                "sentiment": feat.sentiment,
                "word_count": feat.word_count,
                "complexity": feat.complexity,
                "persons": ",".join(feat.entities_persons[:2]),
                "orgs": ",".join(feat.entities_organizations[:2]),
                "locs": ",".join(feat.entities_locations[:2]),
            })
    return pd.DataFrame(rows)


for p in PERSONAS:
    print(f"\n=== Full metadata journey — {p['name']} ===")
    df = full_journey_table(p["user_id"])
    if not df.empty:
        print(df.to_string(index=False))



=== Full metadata journey — Tech & Science Enthusiast ===
 session  rank                               title  score      topic_1  t1_score      topic_2      topic_3 sentiment  word_count  complexity      persons                                     orgs                 locs
       1     1 Открытие экзопланеты телескопом Дже 0.3757        наука    0.4901  образование   технологии   NEUTRAL          45      0.7047  Джеймс Уэбб NASA,европейского космического агентства                Земли
       1     2             Квантовый процессор IBM 0.3702        наука    0.4848   технологии  образование   NEUTRAL          45      0.8116                                                   IBM         США,Германии
       1     3       Рекомендации по здоровому сну 0.3487     здоровье    0.7857 происшествия        наука  POSITIVE          42      0.6149                                                                           
       1     4            Прорыв в лечении диабета 0.3436        наука    0.6

## Step 10. Experiment — tuning `max_topic_ratio`

Зажимаем/ослабляем тематическую дисперсию и смотрим, как меняется feed того же пользователя
с тем же профилем. Все остальные параметры фиксированы.


In [25]:
# Сбрасываем историю и профиль первого пользователя, чтобы эксперимент был воспроизводим.
persona = PERSONAS[0]
uid = persona["user_id"]
pg.rec_history = {(u, c) for (u, c) in pg.rec_history if u != uid}
pg.interactions = [i for i in pg.interactions if i.user_id != uid]
pg.profiles[uid] = create_profile_from_onboarding(uid, persona["onboarding_topics"])

print(f"Exp user: {persona['name']}, onboarding = {persona['onboarding_topics']}\n")

experiment_configs = [
    ("default (0.40, streak=3)", {"max_topic_ratio": 0.40, "max_topic_streak": 3}),
    ("tight (0.20, streak=1)", {"max_topic_ratio": 0.20, "max_topic_streak": 1}),
    ("loose (1.00, streak=99)", {"max_topic_ratio": 1.0, "max_topic_streak": 99}),
]

for name, overrides in experiment_configs:
    pg.rec_history = {(u, c) for (u, c) in pg.rec_history if u != uid}
    feed = generate_feed(pg, uid, feed_size=FEED_SIZE, config={**FEED_CONFIG, **overrides})
    df = feed_summary(feed)
    print(f"\n--- {name} ---")
    topic_seq = " → ".join(r["topic_1"] for r in feed["feed"])
    print(f"topic sequence: {topic_seq}")
    print(df.to_string(index=False))


Exp user: Tech & Science Enthusiast, onboarding = ['технологии', 'наука', 'образование']


--- default (0.40, streak=3) ---
topic sequence: наука → технологии → наука → наука → наука → спорт → происшествия → политика
 rank                                       title  score      topic_1  t1_score sentiment
    1                     Квантовый процессор IBM 0.5702        наука    0.4848   NEUTRAL
    2                      Новый смартфон Яндекса 0.5260   технологии    0.5756   NEUTRAL
    3             Новый ускоритель частиц в ЦЕРНе 0.4519        наука    0.4546   NEUTRAL
    4                    Прорыв в лечении диабета 0.4103        наука    0.6832   NEUTRAL
    5 Открытие экзопланеты телескопом Джеймс Уэбб 0.3757        наука    0.4901   NEUTRAL
    6               Финал Кубка России по футболу 0.3606        спорт    0.9507   NEUTRAL
    7          Исследование вакцины против гриппа 0.3520 происшествия    0.6120   NEUTRAL
    8               Парламентские выборы в Европе 0.3437     по

## Step 11. Experiment — tuning `scoring_weights`

Смотрим, как меняется порядок, если:
- поднять вес `topic_match` c 0.30 до 0.80 → фид станет ещё более «тематическим»
- сделать `freshness` единственным ненулевым → порядок = обратно хронологический

**Важно**: в примере ниже веса не должны суммироваться в 1, так как в scorer'е нет явной нормировки.


In [26]:
scoring_experiments = [
    ("default",
     {**SCORING_WEIGHTS}),
    ("topic-heavy",
     {"topic_match": 0.80, "embedding_sim": 0.10, "entity_match": 0.05,
      "sentiment_match": 0.0, "freshness": 0.05, "format_match": 0.0}),
    ("freshness-only",
     {"topic_match": 0.0, "embedding_sim": 0.0, "entity_match": 0.0,
      "sentiment_match": 0.0, "freshness": 1.0, "format_match": 0.0}),
]

# Сбрасываем профиль и историю
pg.rec_history = {(u, c) for (u, c) in pg.rec_history if u != uid}
pg.interactions = [i for i in pg.interactions if i.user_id != uid]
pg.profiles[uid] = create_profile_from_onboarding(uid, persona["onboarding_topics"])

for name, sw in scoring_experiments:
    pg.rec_history = {(u, c) for (u, c) in pg.rec_history if u != uid}
    cfg = {**FEED_CONFIG, "scoring_weights": sw}
    feed = generate_feed(pg, uid, feed_size=FEED_SIZE, config=cfg)
    print(f"\n--- scoring_weights = {name} ---")
    print(f"weights: {sw}")
    topic_seq = " → ".join(r["topic_1"] for r in feed["feed"])
    print(f"topic sequence: {topic_seq}")
    print(feed_summary(feed, limit=5).to_string(index=False))



--- scoring_weights = default ---
weights: {'topic_match': 0.3, 'embedding_sim': 0.25, 'entity_match': 0.15, 'sentiment_match': 0.05, 'freshness': 0.15, 'format_match': 0.1}
topic sequence: наука → технологии → наука → наука → наука → спорт → происшествия → политика
 rank                                       title  score    topic_1  t1_score sentiment
    1                     Квантовый процессор IBM 0.5702      наука    0.4848   NEUTRAL
    2                      Новый смартфон Яндекса 0.5260 технологии    0.5756   NEUTRAL
    3             Новый ускоритель частиц в ЦЕРНе 0.4519      наука    0.4546   NEUTRAL
    4                    Прорыв в лечении диабета 0.4103      наука    0.6832   NEUTRAL
    5 Открытие экзопланеты телескопом Джеймс Уэбб 0.3757      наука    0.4901   NEUTRAL

--- scoring_weights = topic-heavy ---
weights: {'topic_match': 0.8, 'embedding_sim': 0.1, 'entity_match': 0.05, 'sentiment_match': 0.0, 'freshness': 0.05, 'format_match': 0.0}
topic sequence: наука → нау

## Step 12. Topic distribution evolution — before vs after

Видим, насколько профиль «сместился» в сторону предпочитаемых тем за N сессий.


In [27]:
rows = []
for p in PERSONAS:
    uid = p["user_id"]
    # Снова эмулируем чистый journey, собирая topic_vector после каждой сессии.
    pg.rec_history = {(u, c) for (u, c) in pg.rec_history if u != uid}
    pg.interactions = [i for i in pg.interactions if i.user_id != uid]
    pg.profiles[uid] = create_profile_from_onboarding(uid, p["onboarding_topics"])

    tv0 = pg.profiles[uid].topic_vector.weights.copy()
    rng = random.Random(SEED + hash(p["name"]) % 1000)

    for s in range(N_SESSIONS):
        feed = generate_feed(pg, uid, feed_size=FEED_SIZE)
        simulate_session(pg, uid, feed, p, rng=rng)
    _ = generate_feed(pg, uid, feed_size=0)

    tv_final = pg.profiles[uid].topic_vector.weights

    for t in TOPICS:
        rows.append({
            "persona": p["name"],
            "topic": t,
            "w_initial": round(tv0.get(t, 0.0), 4),
            "w_final": round(tv_final.get(t, 0.0), 4),
            "delta": round(tv_final.get(t, 0.0) - tv0.get(t, 0.0), 4),
            "onboarded": t in p["onboarding_topics"],
            "preferred": t in p["persona_preferred"],
        })

evo_df = pd.DataFrame(rows)

# Для каждой персоны печатаем топ-5 по абсолютной дельте
for p in PERSONAS:
    sub = evo_df[evo_df["persona"] == p["name"]].copy()
    sub["abs_delta"] = sub["delta"].abs()
    sub = sub.sort_values("abs_delta", ascending=False).head(8)
    print(f"\n=== {p['name']}: biggest topic weight shifts ===")
    print(sub[["topic", "w_initial", "w_final", "delta", "onboarded", "preferred"]].to_string(index=False))



=== Tech & Science Enthusiast: biggest topic weight shifts ===
       topic  w_initial  w_final   delta  onboarded  preferred
 образование     0.3175   0.2558 -0.0616       True      False
  технологии     0.3175   0.2614 -0.0561       True       True
       наука     0.3175   0.3597  0.0422       True       True
     финансы     0.0032   0.0385  0.0353      False      False
    здоровье     0.0032   0.0267  0.0235      False      False
      бизнес     0.0032   0.0124  0.0092      False      False
происшествия     0.0032   0.0111  0.0079      False      False
    политика     0.0032   0.0082  0.0050      False      False

=== Sports Fan: biggest topic weight shifts ===
       topic  w_initial  w_final   delta  onboarded  preferred
 развлечения     0.3175   0.2231 -0.0944       True      False
       спорт     0.3175   0.3849  0.0674       True       True
  технологии     0.0032   0.0174  0.0142      False      False
происшествия     0.0032   0.0163  0.0131      False      False
   тр

## Usage Guide — as a playground

### Что можно крутить

| Где | Что | Эффект |
|---|---|---|
| `SIGNAL_WEIGHTS` | веса для LIKE/DISLIKE/BOOKMARK и пороги CLOSE | сила «одобрения» и «отказа» |
| `SCORING_WEIGHTS` | 6 весов компонент | перекос в topic / embedding / freshness |
| `RANKING_PARAMS["max_topic_streak"]` | max подряд одной темы | монотонность фида |
| `RANKING_PARAMS["max_topic_ratio"]` | max доля темы | тематическая насыщенность |
| `RANKING_PARAMS["freshness_halflife_hours"]` | полураспад | влияние «новизны» |
| `PROFILE_PARAMS["learning_rate"]` | 0.08 в проде | скорость адаптации EMA |
| `DEDUP_PARAMS["related_min_gap"]` | 3 в проде | расстановка «похожих» постов |
| `ONBOARDING_PARAMS["baseline_weight"]` | 0.01 в проде | «открытость» новым темам на онбординге |

### Как подставить свой контент

1. Подготовьте CSV c колонками `id,title,body,source_id,source_type,post_date`.
2. Замените `CSV_PATH = PROJECT_ROOT / "rec_playground_sample.csv"`.
3. Если `id` нет — сгенерируйте через `df["id"] = [uuid.uuid4() for _ in range(len(df))]`.
4. Первый прогон запишет `.playground_cache/features_<hash>.pkl` — переиспользуется потом.

### Как добавить новую персону

Просто расширьте `PERSONAS` новым словарём:
```python
PERSONAS.append({
    "user_id": uuid.uuid4(),
    "name": "Business Reader",
    "onboarding_topics": ["бизнес", "финансы", "экономика"],
    "persona_preferred": {"бизнес", "финансы"},
    "persona_avoid": {"развлечения", "криминал"},
    "read_probability": 0.7,
    "like_probability": 0.4,
    "bookmark_probability": 0.3,
    "dislike_probability_offtopic": 0.4,
})
```

### Что не реализовано (упрощения)

| Что | Упрощение в ноутбуке | Prod |
|---|---|---|
| `published_content` маппинг | `post_id == published_id` | отдельная таблица |
| Dedup (EXACT/DUPLICATE) Union-Find | `pg.dedup_clusters = {}` (пусто) | реально считается из similarities |
| RELATED spacing | `pg.related_map = {}` (пусто) | граф из similarities |
| Redis debounce recommendations.updated | — | 15-мин окно |
| APScheduler jobs | вручную через `generate_feed()` | автоматически каждые N минут |
| Kafka consumer для user.interactions.batch | прямой вызов симулятора | реальный консьюмер |
| Cold-start trending cache | — | Redis TTL 1h |

Все *формулы*, *веса*, *пороги*, *EMA*, *diversity constraints* — один-в-один как в prod.

### Оценка качества

Для наглядных метрик:
- **Topic alignment**: какая доля постов в финальном фиде ∈ `preferred_topics`.
- **Topic vector shift**: L1-расстояние между `profile.topic_vector` до/после.
- **Coverage**: сколько уникальных постов юзер увидел в N сессиях.
- **Diversity**: Gini coefficient по топикам в финальном фиде.


In [28]:
# === Квик-метрики качества на последнем journey ===
print("=" * 70)
print("PLAYGROUND SUMMARY")
print("=" * 70)
for p in PERSONAS:
    uid = p["user_id"]
    prof = pg.profiles[uid]
    feed = generate_feed(pg, uid, feed_size=FEED_SIZE, respect_history=False)
    topic_counts = defaultdict(int)
    for r in feed["feed"]:
        topic_counts[r["topic_1"]] += 1
    preferred_hit = sum(
        topic_counts[t] for t in p["persona_preferred"] if t in topic_counts
    )
    total = sum(topic_counts.values())
    alignment = preferred_hit / total if total else 0.0
    print(f"\n[{p['name']}]")
    print(f"  interaction_count: {prof.interaction_count}")
    print(f"  embedding set: {prof.embedding is not None}")
    print(f"  topic_1 distribution in final feed: {dict(topic_counts)}")
    print(f"  preferred-topic alignment: {alignment:.2%}")
    print(f"  top-3 topic_vector weights: "
          f"{sorted(prof.topic_vector.weights.items(), key=lambda kv: -kv[1])[:3]}")

print("\n" + "=" * 70)
print("Готово! Меняйте константы CONFIG сверху и перезапускайте — рекомендации переедут.")


PLAYGROUND SUMMARY

[Tech & Science Enthusiast]
  interaction_count: 34
  embedding set: True
  topic_1 distribution in final feed: {'наука': 4, 'технологии': 1, 'происшествия': 1, 'бизнес': 1, 'политика': 1}
  preferred-topic alignment: 62.50%
  top-3 topic_vector weights: [('наука', 0.3596832828140795), ('технологии', 0.26140973229592585), ('образование', 0.2558406539682468)]

[Sports Fan]
  interaction_count: 28
  embedding set: True
  topic_1 distribution in final feed: {'спорт': 3, 'бизнес': 1, 'здоровье': 1, 'происшествия': 1, 'наука': 1, 'политика': 1}
  preferred-topic alignment: 50.00%
  top-3 topic_vector weights: [('спорт', 0.38487820827576796), ('здоровье', 0.3085641867696876), ('развлечения', 0.22308949202528444)]

[Health Conscious]
  interaction_count: 33
  embedding set: True
  topic_1 distribution in final feed: {'наука': 4, 'бизнес': 1, 'происшествия': 1, 'здоровье': 1, 'политика': 1}
  preferred-topic alignment: 62.50%
  top-3 topic_vector weights: [('наука', 0.34065